# 🧹 高级专家 机考模拟练习一：数据清洗与特征处理 (Pandas)

> **🎯 考核目标**：在不涉及到分布式的单机场景下，考核候选人对 Pandas 高级 API 的熟练度（避免使用低效的 for 循环），以及对业务异常数据的敏锐度。

---

## 🧪 试题一：海量数据框的极速缺失值填充

**📝 你的任务**：
给定一个包含各种异常和缺失的数据框 `df`。
请写出**一行代码**或最高效的逻辑完成以下操作：
1. 对所有**数值型**特征，使用该列的中位数 (median) 进行填充。
2. 对所有**分类型**（如 object/category）特征，使用该列的众数 (mode) 进行填充。
3. **强烈禁止**对列进行 `for` 循环遍历判断类型！

In [12]:
import pandas as pd
import numpy as np

# 构建模拟数据
df = pd.DataFrame({
    'age': [25, np.nan, 30, 22, np.nan],
    'salary': [10000, 15000, np.nan, 8000, 12000],
    'gender': ['M', 'F', np.nan, 'F', 'M'],
    'city': ['BJ', np.nan, 'SH', 'BJ', 'GZ']
})

# ==================
# 你的代码写在这里：

# 数值型填充逻辑：
df_num = df.select_dtypes(include='number').columns
df[df_num] = df[df_num].fillna(df[df_num].median())
df_obj = df.select_dtypes(include='str').columns
df[df_obj] = df[df_obj].fillna(df[df_obj].mode().iloc[0])
df
# ==================

# ---------------------
# 💡 面试官答疑：为什么要这么做？
# 高级专家 标准要求代码足够 Pythonic 且高效。for 循环在 Python 中极其缓慢，而 Pandas 底层是 C++ 优化的矢量化计算。
# 标准答案运用了 select_dtypes 的技巧：
# df[df.select_dtypes(include=['number']).columns] = df.select_dtypes(include=['number']).apply(lambda x: x.fillna(x.median()))
# df[df.select_dtypes(include=['object']).columns] = df.select_dtypes(include=['object']).apply(lambda x: x.fillna(x.mode()[0]))

,age,salary,gender,city
0,25.0,10000.0,M,BJ
1,25.0,15000.0,F,BJ
2,30.0,11000.0,F,SH
3,22.0,8000.0,F,BJ
4,25.0,12000.0,M,GZ


## 🧪 试题二：分箱与异常值剔除

**📝 你的任务**：
1. 剔除极其离谱的异常订单金额（剔除金额在 99% 分位数以上的订单作为异常值）。
2. 将用户年龄按指定的业务切片（0-18, 19-25, 26-35, 36-45, 46+）进行完美分箱，不能写臃肿的 `if...else`。
3. 计算分段后每组的平均购买金额和用户数量。

In [ ]:
# 构建模拟数据
np.random.seed(42)
df_orders = pd.DataFrame({
    'user_id': range(1, 1001),
    'age': np.random.randint(15, 65, 1000),
    'purchase_amount': np.append(np.random.normal(500, 100, 990), np.random.uniform(5000, 10000, 10)) # 混入极其离谱的高常值
})

# ==================
# 你的代码写在这里：
df_orders.describe(include='all')
# 1. 剔除 99% 分位数以上异常值
df_clean = df_orders[df_orders['purchase_amount']<=df_orders['purchase_amount'].quantile(0.99)]

# 2. pd.cut 优雅分箱
df_clean['age_range'] = pd.cut(df_clean['age'],bins=[0,18,25,35,45,100])
# 3. 分组聚合 (groupby + agg)
df_clean.groupby('age_range')[['user_id','purchase_amount']].agg(
    avg_amount = ('purchase_amount','mean'),
    user_cnt = ('user_id','count')
)
# ==================

# ---------------------
# 💡 最佳实践防守：
# 阈值：p99 = df_orders['purchase_amount'].quantile(0.99)
# 剔除：df_clean = df_orders[df_orders['purchase_amount'] <= p99]
# 分箱：df_clean['age_group'] = pd.cut(df_clean['age'], bins=[0, 18, 25, 35, 45, float('inf')], labels=['0-18', '19-25', '26-35', '36-45', '46+'])
# 聚合：df_clean.groupby('age_group')['purchase_amount'].agg(['mean', 'count'])




,avg_amount,user_cnt
age_range,,
"(0, 18]",519.071473,83
"(18, 25]",500.003333,123
"(25, 35]",514.686375,182
"(35, 45]",497.855008,215
"(45, 100]",510.928696,387
